In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import sys
import os
from datetime import datetime

project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.utils.team_info import *
from src.utils.helper_functions import *
from src.pipeline.props_pipeline.ppm_pipeline import *
from src.pipeline.props_pipeline.apm_pipeline import *
from src.pipeline.props_pipeline.rpm_pipeline import *
from src.pipeline.props_pipeline.min_pipeline import *
from src.live import *
from src.historical_analysis.dataScraper import *

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

### Get updated lineups

In [2]:
from src.utils.scrap_starters import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
print("\nQuestionable Players:")
print(scraper.getQuestionablePlayers())
print("\nOut Players:")
print(scraper.getOutPlayers())
outPlayers = scraper.getOutPlayers()
scraper.updateTeamInfo()  # Update teamInfo.py


Questionable Players:
No data available. Run getDict() first.
{}

Out Players:
No data available. Run getDict() first.
{}
No data available. Run getDict() first.
No data available. Run getDict() first.


### Dataset

In [4]:
s25 = pd.read_csv('data/raw/season_stats/S25.csv').sort_values(by='GAME_DATE')
s26 = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')
p26 = pd.read_csv('data/raw/playoff_stats/P26.csv').sort_values(by='GAME_DATE')
base_df = pd.concat([s26, p26])
base_df.tail()

,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,START_POSITION,SPD,DIST,ORBC,DRBC,RBC,TCHS,SAST,FTAST,PASS,CFGM,CFGA,CFG_PCT,UFGM,UFGA,UFG_PCT,DFGM,DFGA,DFG_PCT,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,IS_PLAYOFF,POS,AGE,name
57,57,2025-26,1630540,Miles McBride,Miles,1610612752,NYK,New York Knicks,42500125,2026-04-28T00:00:00,NYK vs. ATL,W,20.033333,0,4,0.000,0,2,0.000,0,0,0.000,1,2,3,1,1,0,0,1,2,0,0,19,4.1,0,0,4.0,1,20:02,1,143.5,142.9,142.9,101.5,100.0,100.0,42.1,42.9,42.9,0.05,1.0,16.7,0.063,0.087,0.077,16.7,16.7,0.000,0.000,0.104,0.109,98.48,99.43,82.86,99.43,-0.043,42,0.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45,79,0.570,11,26,0.423,25,34,0.735,8,40,48,26,11.0,6,6,2,23,26,126,29.0,130.0,134.0,101.6,103.2,28.4,30.9,0.578,2.36,19.5,0.325,0.824,0.604,0.117,0.639,0.670,96.2,94.0,78.33,94,0.661,1610612737,ATL,Atlanta Hawks,37,83,0.446,13,42,0.310,10,17,0.588,5,22,27,25,10.0,7,2,6,26,23,97,-29.0,101.6,103.2,130.0,134.0,-28.4,-30.9,0.676,2.50,19.5,0.176,0.675,0.396,0.106,0.524,0.536,96.2,94.0,78.33,94,0.339,1,SG,25.0,Miles McBride
58,58,2025-26,1631248,Baylor Scheierman,Baylor,1610612738,BOS,Boston Celtics,42500115,2026-04-28T00:00:00,BOS vs. PHI,L,8.650000,1,2,0.500,1,2,0.500,0,0,0.000,0,0,0,0,1,0,0,0,1,0,3,-3,2.0,0,0,4.0,1,8:39,1,101.6,83.3,83.3,114.2,105.9,105.9,-12.6,-22.5,-22.5,0.00,0.0,0.0,0.000,0.000,0.000,33.3,33.3,0.750,0.750,0.136,0.160,84.68,97.11,80.92,97.11,0.000,18,1.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,36,89,0.404,11,39,0.282,14,23,0.609,15,34,49,25,12.0,6,0,3,17,20,97,-16.0,100.9,103.2,121.3,120.2,-20.4,-17.0,0.694,2.08,18.4,0.300,0.829,0.515,0.128,0.466,0.489,94.6,94.0,78.33,94,0.404,1610612755,PHI,Philadelphia 76ers,39,78,0.500,15,42,0.357,20,23,0.870,5,42,47,24,10.0,5,3,0,20,17,113,16.0,121.3,120.2,100.9,103.2,20.4,17.0,0.615,2.40,19.2,0.171,0.700,0.485,0.106,0.596,0.641,94.6,94.0,78.33,94,0.596,1,SG,25.0,Baylor Scheierman
59,59,2025-26,1642278,Tyler Kolek,Tyler,1610612752,NYK,New York Knicks,42500125,2026-04-28T00:00:00,NYK vs. ATL,W,3.466667,0,1,0.000,0,0,0.000,0,0,0.000,0,1,1,2,0,0,0,0,0,0,0,2,4.2,0,0,3.0,1,3:28,1,147.9,125.0,125.0,100.0,100.0,100.0,47.9,25.0,25.0,0.50,0.0,66.7,0.000,0.250,0.143,0.0,0.0,0.000,0.000,0.111,0.129,102.18,110.77,92.31,110.77,0.100,8,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45,79,0.570,11,26,0.423,25,34,0.735,8,40,48,26,11.0,6,6,2,23,26,126,29.0,130.0,134.0,101.6,103.2,28.4,30

### Load latest odds on file

In [5]:
def get_latest_file(pattern):
    files = list(Path('data/raw/team_lines').glob(pattern))
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

file = get_latest_file('NBA_*.json')
if file is None:
    raise ValueError("No JSON file found")

# try normal load first
try:
    team_dds = pd.read_json(file)
except ValueError:
    # fallback for nested JSON
    import json
    with open(file) as f:
        data = json.load(f)
    team_dds = pd.json_normalize(data)

print("Loaded:", file.name)
team_dds.head()

Loaded: NBA_20260428_230331.json


,home_team,away_team,commence_time,bookmakers
0,Detroit Pistons,Orlando Magic,2026-04-29 23:10:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
1,Cleveland Cavaliers,Toronto Raptors,2026-04-29 23:40:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
2,Los Angeles Lakers,Houston Rockets,2026-04-30 02:10:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
3,Atlanta Hawks,New York Knicks,2026-04-30 23:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
4,Philadelphia 76ers,Boston Celtics,2026-05-01 00:10:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."


In [24]:
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

us_file = get_latest_file(f'NBA_US_{today}*.csv')
dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

#load season stats
pts_df = pd.read_csv('data/processed/training/S26_TRAINING_PPM.csv')
ast_df = pd.read_csv('data/processed/training/S26_TRAINING_APM.csv')
reb_df = pd.read_csv('data/processed/training/S26_TRAINING_RPM.csv')
min_df = pd.read_csv('data/processed/training/S26_TRAINING_MIN.csv')
pts_ast_df = pd.read_csv('data/processed/training/S26_TRAINING_PAPM.csv')

#load dfs lines
lines_dfs = pd.read_csv(dfs_file)
lines_dfs = lines_dfs[lines_dfs['COMMENCE_TIME'] == '2026-04-29']
lines_dfs_pts = lines_dfs[(lines_dfs['CATEGORY'] == 'player_points')]
lines_dfs_ast = lines_dfs[(lines_dfs['CATEGORY'] == 'player_assists')]
lines_dfs_reb = lines_dfs[(lines_dfs['CATEGORY'] == 'player_rebounds')]
pts_names = lines_dfs_pts['NAME'].unique()
ast_names = lines_dfs_ast['NAME'].unique()
reb_names = lines_dfs_reb['NAME'].unique()

#load us lines with actual odds
lines_us = pd.read_csv(us_file)
lines_us = lines_us[lines_us['COMMENCE_TIME'] == '2026-04-29']
lines_us_pts = lines_us[(lines_us['CATEGORY'] == 'player_points')]
lines_us_ast = lines_us[(lines_us['CATEGORY'] == 'player_assists')]
lines_us_reb = lines_us[(lines_us['CATEGORY'] == 'player_rebounds')]
print(f"DFS latest pull: {lines_dfs['DATA_PULLED_AT'].max()}")
print(f"US latest pull: {lines_us['DATA_PULLED_AT'].max()}")

lines_dfs_pts.head()

DFS latest pull: 2026-04-28 23:03:31
US latest pull: 2026-04-28 23:03:03


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,PrizePicks,player_points,Cade Cunningham,Over,28.5,-137,2026-04-29,2026-04-29T06:03:10Z,2026-04-28 23:03:31
1,PrizePicks,player_points,Cade Cunningham,Under,28.5,-137,2026-04-29,2026-04-29T06:03:10Z,2026-04-28 23:03:31
2,PrizePicks,player_points,Paolo Banchero,Over,21.5,-137,2026-04-29,2026-04-29T06:03:10Z,2026-04-28 23:03:31
3,PrizePicks,player_points,Paolo Banchero,Under,21.5,-137,2026-04-29,2026-04-29T06:03:10Z,2026-04-28 23:03:31
4,PrizePicks,player_points,Desmond Bane,Over,18.5,-137,2026-04-29,2026-04-29T06:03:10Z,2026-04-28 23:03:31


### Load my models

In [25]:
import joblib

#minutes
min_bundle = joblib.load("src/models/saved_models/min_quantile_xgb_2026-01-02.joblib")
min_quantile_models = min_bundle["quantile_models"]
min_feature_names = min_bundle["feature_names"]

#points per minute
ppm_bundle = joblib.load("src/models/saved_models/ppm_quantile_xgb_2026-01-01.joblib")
ppm_quantile_models = ppm_bundle["quantile_models"]
ppm_feature_names = ppm_bundle["feature_names"]

#assists per minute
apm_bundle = joblib.load("src/models/saved_models/apm_quantile_xgb_2026-01-01.joblib")
apm_quantile_models = apm_bundle["quantile_models"]
apm_feature_names = apm_bundle["feature_names"]

#rebounds per minute
rpm_bundle = joblib.load("src/models/saved_models/rpm_quantile_xgb_2026-01-01.joblib")
rpm_quantile_models = rpm_bundle["quantile_models"]
rpm_feature_names = rpm_bundle["feature_names"]

### Get Min predictions and Stat Per Min predictions 

In [26]:
pts_preds = predict_min_times_rate(
    pts_names, min_df, pts_df, current_date,
    rate_pipeline=ppm_pipeline,
    rate_quantile_models=ppm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="PTS",
)
ast_preds = predict_min_times_rate(
    ast_names, min_df, ast_df, current_date,
    rate_pipeline=apm_pipeline,
    rate_quantile_models=apm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="AST",
)
reb_preds = predict_min_times_rate(
    reb_names, min_df, reb_df, current_date,
    rate_pipeline=rpm_pipeline,
    rate_quantile_models=rpm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="REB",
)
pts_preds.head(10)

[SKIP] Wendell Carter Jr: min_pipeline returned None (need >= 10 games)
[SKIP] R.J. Barrett: min_pipeline returned None (need >= 10 games)
[SKIP] R.J. Barrett: min_pipeline returned None (need >= 10 games)
[SKIP] Wendell Carter Jr: min_pipeline returned None (need >= 10 games)
[SKIP] R.J. Barrett: min_pipeline returned None (need >= 10 games)


,PLAYER_NAME,MARKET,MIN_Q10,MIN_Q50,MIN_Q90,RATE_Q10,RATE_Q50,RATE_Q90,STAT_Q10,STAT_Q50,STAT_Q90,RATE_HISTORY
0,Cade Cunningham,PTS,29.21,37.78,42.31,0.3913,0.6315,0.8870,11.43,23.86,37.53,"[0.7754010695187166, 0.2580645161290322, 0.660..."
1,Paolo Banchero,PTS,29.29,38.11,41.49,0.3956,0.5835,0.8479,11.59,22.24,35.18,"[1.0617059891107077, 0.9007506255212676, 0.796..."
2,Desmond Bane,PTS,25.05,34.17,39.90,0.2799,0.4845,0.7472,7.01,16.56,29.81,"[0.5233453052847614, 0.467032967032967, 0.6616..."
3,Tobias Harris,PTS,23.46,33.52,39.83,0.2300,0.4548,0.6872,5.40,15.25,27.37,"[0.3186646433990895, 0.5172413793103449, 0.438..."
4,Franz Wagner,PTS,22.54,31.28,37.21,0.3934,0.6129,0.8687,8.87,19.17,32.32,"[0.9393346379647748, 0.9610983981693364, 0.674..."
5,Jalen Duren,PTS,19.68,26.37,33.24,0.2290,0.4488,0.6956,4.51,11.83,23.12,"[0.968392737054472, 1.07326178254783, 0.6, 0.6..."
6,Jalen Suggs,PTS,23.98,32.96,39.67,0.2250,0.5059,0.7912,5.39,16.68,31.39,"[0.4109589041095891, 0.4081632653061224, 0.235..."
7,Ausar Thompson,PTS,21.36,29.86,38.30,0.1310,0.3360,0.5833,2.80,10.03,22.34,"[0.1674730182359508, 0.4855460144764645, 0.379..."
8,Duncan Robinson,PTS,20.03,28.59,35.62,0.1420,0.3639,0.6117,2.84,10.40,21.79,"[0.3041825095057034, 0.5191594561186651, 0.378..."
9,Anthony Black,PTS,17.33,23.00,29.45,0.1795,0.4151,0.6589,3.11,9.55,19.40,"[0.5257836198179979, 0.6417112299465241, 0.222..."


### Get Line Probabilities

In [27]:
all_line_probs = pd.concat([
    line_probs_for_market(ast_preds, lines_dfs_ast, run_pts_simulation),
    line_probs_for_market(reb_preds, lines_dfs_reb, run_pts_simulation),
    line_probs_for_market(pts_preds, lines_dfs_pts, run_pts_simulation),
], ignore_index=True)
all_line_probs.sample(10)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER
4,James Harden,AST,6.5,29.29,38.35,42.79,2.95,7.21,11.86,0.493,0.507
38,Jalen Suggs,PTS,13.5,23.98,32.96,39.67,5.39,16.68,31.39,0.715,0.285
54,Jakob Poeltl,PTS,6.5,14.60,20.61,29.41,2.46,7.38,17.98,0.653,0.346
10,Brandon Ingram,AST,3.5,27.75,36.46,41.20,0.97,3.11,6.48,0.472,0.527
42,Javonte Green,PTS,5.5,16.12,20.57,27.41,1.29,5.63,15.21,0.714,0.286
11,Jakob Poeltl,AST,1.5,14.60,20.61,29.41,0.00,1.58,4.32,0.678,0.322
9,Paolo Banchero,AST,4.5,29.29,38.11,41.49,2.39,5.71,10.63,0.781,0.219
39,Ausar Thompson,PTS,10.5,21.36,29.86,38.30,2.80,10.03,22.34,0.582,0.418
25,Donovan Mitchell,REB,4.5,28.20,36.49,41.41,1.60,4.70,8.98,0.670,0.330
55,Jamal Shead,PTS,6.5,20.64,30.07,37.97,1.83,7.75,19.57,0.572,0.428


In [28]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='Underdog',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

underdog_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
underdog_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
44,Donovan Mitchell,PTS,26.5,28.20,36.49,41.41,13.06,26.04,40.59,0.568,0.432,PTS,Underdog,Toronto Raptors,-9.0,216.0,112.1,5.0,99.22,21.0,-113.0,-104.0,0.531,0.510,24.1,27.5,10.87,-2.4,1.0,0.221,0.413,0.587,-22.15,15.14,0.6,0.5,0.53,0.61,33.03,3.83,0.29,0.07,24.17,6.0
49,Collin Murray-Boyles,PTS,12.5,17.23,22.36,30.34,3.37,9.88,20.97,0.585,0.415,PTS,Underdog,Cleveland Cavaliers,9.0,216.0,114.1,15.0,100.70,13.0,-105.0,-114.0,0.512,0.533,14.4,16.0,6.28,1.9,3.5,-0.303,0.619,0.381,20.85,-28.48,0.8,0.7,0.53,0.25,22.41,4.60,0.20,0.08,13.00,6.0
51,Max Strus,PTS,8.5,17.68,23.69,30.43,2.83,9.70,20.56,0.623,0.377,PTS,Underdog,Toronto Raptors,-9.0,216.0,112.1,5.0,99.22,21.0,-118.0,-105.0,0.541,0.512,10.5,9.0,8.28,2.0,0.5,-0.242,0.596,0.404,10.11,-21.12,0.6,0.5,0.47,0.50,24.16,3.79,0.17,0.05,11.50,4.0
45,James Harden,PTS,20.5,29.29,38.35,42.79,10.45,21.72,35.37,0.696,0.304,PTS,Underdog,Toronto Raptors,-9.0,216.0,112.1,5.0,99.22,21.0,-106.0,-114.0,0.515,0.533,20.5,19.5,4.65,0.0,-1.0,0.000,0.500,0.500,-2.83,-6.14,0.4,0.4,0.40,0.57,33.46,4.20,0.27,0.06,23.60,5.0
48,Evan Mobley,PTS,16.5,25.31,33.47,38.99,7.35,16.44,29.34,0.611,0.389,PTS,Underdog,Toronto Raptors,-9.0,216.0,112.1,5.0,99.22,21.0,-114.0,100.0,0.533,0.500,17.3,16.0,8.86,0.8,-0.5,-0.090,0.536,0.464,0.62,-7.20,0.4,0.5,0.60,0.59,29.84,5.06,0.22,0.07,16.43,7.0


In [29]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='PrizePicks',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

prizePicks_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
prizePicks_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
25,Donovan Mitchell,REB,4.5,28.20,36.49,41.41,1.60,4.70,8.98,0.670,0.330,REB,PrizePicks,Toronto Raptors,-9.0,216.0,112.1,5.0,99.22,21.0,-137.0,-137.0,0.578,0.578,4.9,5.5,1.79,0.9,1.5,-0.503,0.693,0.307,19.88,-46.89,0.8,0.6,0.60,0.47,33.03,3.83,0.29,0.07,4.67,6.0
18,Evan Mobley,REB,8.5,25.31,33.47,38.99,3.97,8.95,15.09,0.511,0.489,REB,PrizePicks,Toronto Raptors,-9.0,216.0,112.1,5.0,99.22,21.0,-120.0,108.0,0.545,0.481,9.2,7.5,4.96,0.7,-1.0,-0.141,0.556,0.444,1.93,-7.65,0.2,0.4,0.47,0.51,29.84,5.06,0.22,0.07,7.43,7.0
15,Desmond Bane,REB,4.0,25.05,34.17,39.90,1.25,4.30,8.55,0.646,0.353,REB,PrizePicks,Detroit Pistons,10.0,211.5,108.9,2.0,99.88,19.0,-137.0,-137.0,0.578,0.578,4.5,4.5,2.07,0.5,0.5,-0.242,0.596,0.404,3.10,-30.11,0.6,0.5,0.47,0.42,30.77,7.41,0.21,0.05,5.25,8.0
57,Dean Wade,PTS,4.5,14.81,21.26,28.55,0.59,4.47,12.93,0.543,0.457,PTS,PrizePicks,Toronto Raptors,-9.0,216.0,112.1,5.0,99.22,21.0,-121.0,102.0,0.548,0.495,4.9,5.0,2.88,0.4,0.5,-0.139,0.555,0.445,1.37,-10.11,0.6,0.6,0.60,0.60,22.77,3.66,0.08,0.04,4.50,6.0
52,Ja'Kobe Walter,PTS,7.5,18.24,24.03,31.30,1.78,7.20,17.55,0.583,0.417,PTS,PrizePicks,Cleveland Cavaliers,9.0,216.0,114.1,15.0,100.70,13.0,-104.0,-105.0,0.510,0.512,8.6,8.0,5.64,1.1,0.5,-0.195,0.577,0.423,13.18,-17.41,0.4,0.5,0.53,0.41,27.37,2.27,0.13,0.03,4.71,7.0


In [30]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='Betr DFS',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

betr_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
betr_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
39,Ausar Thompson,PTS,10.5,21.36,29.86,38.30,2.80,10.03,22.34,0.582,0.418,PTS,Betr DFS,Orlando Magic,-10.0,211.5,113.6,13.0,100.56,14.0,-105.0,-103.0,0.512,0.507,10.0,9.0,3.53,-0.5,-1.5,0.142,0.444,0.556,-13.31,9.58,0.4,0.4,0.40,0.39,28.02,4.70,0.15,0.05,9.88,8.0
15,Desmond Bane,REB,4.0,25.05,34.17,39.90,1.25,4.30,8.55,0.646,0.353,REB,Betr DFS,Detroit Pistons,10.0,211.5,108.9,2.0,99.88,19.0,-137.0,-137.0,0.578,0.578,4.5,4.5,2.07,0.5,0.5,-0.242,0.596,0.404,3.10,-30.11,0.6,0.5,0.47,0.42,30.77,7.41,0.21,0.05,5.25,8.0
47,Brandon Ingram,PTS,19.5,27.75,36.46,41.20,8.83,18.56,32.27,0.441,0.559,PTS,Betr DFS,Cleveland Cavaliers,9.0,216.0,114.1,15.0,100.70,13.0,102.0,-125.0,0.495,0.556,19.3,17.0,8.53,-0.2,-2.5,0.023,0.491,0.509,-0.82,-8.38,0.4,0.4,0.33,0.57,32.96,3.68,0.24,0.05,18.14,7.0
55,Jamal Shead,PTS,6.5,20.64,30.07,37.97,1.83,7.75,19.57,0.572,0.428,PTS,Betr DFS,Cleveland Cavaliers,9.0,216.0,114.1,15.0,100.70,13.0,-112.0,-111.0,0.528,0.526,6.4,5.5,4.72,-0.1,-1.0,0.021,0.492,0.508,-6.87,-3.43,0.2,0.4,0.47,0.45,26.30,4.94,0.14,0.05,7.14,7.0
40,Duncan Robinson,PTS,9.5,20.03,28.59,35.62,2.84,10.40,21.79,0.745,0.255,PTS,Betr DFS,Orlando Magic,-10.0,211.5,113.6,13.0,100.56,14.0,-110.0,-108.0,0.524,0.519,12.9,10.5,4.89,3.4,1.0,-0.695,0.756,0.244,44.33,-53.01,0.4,0.7,0.80,0.63,26.76,3.09,0.17,0.05,8.57,7.0


In [31]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='DraftKings Pick6',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

draftKings_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
draftKings_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
44,Donovan Mitchell,PTS,26.5,28.20,36.49,41.41,13.06,26.04,40.59,0.568,0.432,PTS,DraftKings Pick6,Toronto Raptors,-9.0,216.0,112.1,5.0,99.22,21.0,-113.0,-104.0,0.531,0.510,24.1,27.5,10.87,-2.4,1.0,0.221,0.413,0.587,-22.15,15.14,0.6,0.5,0.53,0.61,33.03,3.83,0.29,0.07,24.17,6.0
46,Scottie Barnes,PTS,19.5,28.08,36.96,41.31,7.19,17.65,30.14,0.581,0.419,PTS,DraftKings Pick6,Cleveland Cavaliers,9.0,216.0,114.1,15.0,100.70,13.0,-104.0,-107.0,0.510,0.517,19.4,19.5,7.56,-0.1,0.0,0.013,0.495,0.505,-2.90,-2.30,0.8,0.5,0.47,0.38,32.52,5.74,0.24,0.06,23.29,7.0
45,James Harden,PTS,20.5,29.29,38.35,42.79,10.45,21.72,35.37,0.696,0.304,PTS,DraftKings Pick6,Toronto Raptors,-9.0,216.0,112.1,5.0,99.22,21.0,-106.0,-114.0,0.515,0.533,20.5,19.5,4.65,0.0,-1.0,0.000,0.500,0.500,-2.83,-6.14,0.4,0.4,0.40,0.57,33.46,4.20,0.27,0.06,23.60,5.0
1,Jalen Suggs,AST,4.5,23.98,32.96,39.67,1.74,4.73,9.63,0.535,0.465,AST,DraftKings Pick6,Detroit Pistons,10.0,211.5,108.9,2.0,99.88,19.0,-118.0,-102.0,0.541,0.505,5.3,4.0,3.23,0.8,-0.5,-0.248,0.598,0.402,10.48,-20.39,0.0,0.3,0.47,0.52,31.87,5.37,0.21,0.05,4.38,8.0
51,Max Strus,PTS,8.5,17.68,23.69,30.43,2.83,9.70,20.56,0.623,0.377,PTS,DraftKings Pick6,Toronto Raptors,-9.0,216.0,112.1,5.0,99.22,21.0,-118.0,-105.0,0.541,0.512,10.5,9.0,8.28,2.0,0.5,-0.242,0.596,0.404,10.11,-21.12,0.6,0.5,0.47,0.50,24.16,3.79,0.17,0.05,11.50,4.0


In [32]:
all_line_probs = pd.concat([underdog_all_lines, prizePicks_all_lines, betr_all_lines, draftKings_all_lines])
all_line_probs.to_json('data/props/ev_analysis/all_line_probs.json', orient='records', lines=True)
all_line_probs.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
40,Duncan Robinson,PTS,9.5,20.03,28.59,35.62,2.84,10.40,21.79,0.745,0.255,PTS,PrizePicks,Orlando Magic,-10.0,211.5,113.6,13.0,100.56,14.0,-110.0,-108.0,0.524,0.519,12.9,10.5,4.89,3.4,1.0,-0.695,0.756,0.244,44.33,-53.01,0.4,0.7,0.80,0.63,26.76,3.09,0.17,0.05,8.57,7.0
47,Brandon Ingram,PTS,19.5,27.75,36.46,41.20,8.83,18.56,32.27,0.441,0.559,PTS,Betr DFS,Cleveland Cavaliers,9.0,216.0,114.1,15.0,100.70,13.0,102.0,-125.0,0.495,0.556,19.3,17.0,8.53,-0.2,-2.5,0.023,0.491,0.509,-0.82,-8.38,0.4,0.4,0.33,0.57,32.96,3.68,0.24,0.05,18.14,7.0
1,Jalen Suggs,AST,4.5,23.98,32.96,39.67,1.74,4.73,9.63,0.535,0.465,AST,PrizePicks,Detroit Pistons,10.0,211.5,108.9,2.0,99.88,19.0,-118.0,-102.0,0.541,0.505,5.3,4.0,3.23,0.8,-0.5,-0.248,0.598,0.402,10.48,-20.39,0.0,0.3,0.47,0.52,31.87,5.37,0.21,0.05,4.38,8.0
22,Brandon Ingram,REB,5.5,27.75,36.46,41.20,1.72,5.14,9.17,0.397,0.603,REB,DraftKings Pick6,Cleveland Cavaliers,9.0,216.0,114.1,15.0,100.70,13.0,128.0,-145.0,0.439,0.592,5.1,5.0,2.23,-0.4,-0.5,0.179,0.429,0.571,-2.19,-3.52,0.4,0.5,0.47,0.57,32.96,3.68,0.24,0.05,5.00,7.0
12,Paolo Banchero,REB,8.5,29.29,38.11,41.49,3.29,8.21,14.25,0.590,0.410,REB,PrizePicks,Detroit Pistons,10.0,211.5,108.9,2.0,99.88,19.0,-137.0,-137.0,0.578,0.578,9.0,8.5,3.16,1.0,0.5,-0.316,0.624,0.376,7.95,-34.95,0.6,0.5,0.47,0.49,35.04,4.49,0.26,0.04,8.86,7.0


In [33]:
all_line_probs[all_line_probs['PLAYER_NAME'] == 'Cade Cunningham']

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES


### Get top EVs for 2 legs

In [34]:
slate_path = build_greedy_slate(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks.json",
)
print(slate_path)

Legs: 35  |  Pairs: 10  |  Slate: 2  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks.json


In [35]:
slate_path = build_greedy_slate(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog.json",
)
print(slate_path)

Legs: 14  |  Pairs: 3  |  Slate: 1  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog.json


In [36]:
slate_path = build_greedy_slate(
    prob_df=draftKings_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/draftKings.json",
)
print(slate_path)

Legs: 35  |  Pairs: 17  |  Slate: 3  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings.json


In [37]:
slate_path = build_greedy_slate(
    prob_df=betr_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/betr.json",
)
print(slate_path)

Legs: 34  |  Pairs: 8  |  Slate: 2  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr.json


### Top EVs for 3 Legs

In [38]:
slate_path = build_greedy_slate_3leg(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks_3leg.json",
)
print(slate_path)

Legs: 35  |  Triples: 28  |  Slate: 2  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks_3leg.json


In [39]:
slate_path = build_greedy_slate_3leg(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog_3leg.json",
)
print(slate_path)

Legs: 14  |  Triples: 2  |  Slate: 1  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog_3leg.json


In [40]:
slate_path = build_greedy_slate_3leg(
    prob_df=betr_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/betr_3leg.json",
)
print(slate_path)

Legs: 34  |  Triples: 27  |  Slate: 2  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr_3leg.json


In [41]:
slate_path = build_greedy_slate_3leg(
    prob_df=draftKings_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/draftKings_3leg.json",
)
print(slate_path)

Legs: 35  |  Triples: 43  |  Slate: 2  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings_3leg.json
